# Phase 3 — Bayesian Structural Recommendation Engine

**Build order (start simple, add structure):** this notebook is step 1–2 — scope the DAG, then build a *baseline* hierarchical model with country partial-pooling on the treatment effect. Later steps add country/year effects, the mechanism layer, and counterfactual simulation.

## Step 1 — Structural DAG (scope)

The end-state wants a *mechanism breakdown* ("tax X%, switching Y%, ..."), which a reduced-form single coefficient can't give. The full causal chain is:

```
policy  ->  energy prices  ->  fuel mix  ->  emissions
 HAVE        MISSING            HAVE          HAVE
```

- **policy:** `has_tax`, `has_ets`, `tax_price_only`, `ets_price_only`, `fuel_subsidy_gdp`
- **energy prices:** NOT measured (we have *carbon* prices = the policy lever, not the retail energy price firms/households face). This link **collapses** to reduced-form.
- **fuel mix:** `fossil_pct_filled`, `renewable_pct`, `nuclear_pct`, `energy_per_capita`, and emissions split by fuel (`coal/gas/oil_co2_per_capita`).
- **emissions:** `co2_per_capita_future_trend` (3-yr forward).

Realistic skeleton: a two-link chain `policy -> fuel mix -> emissions`, decomposed via the **Kaya identity** `CO2/pop = (GDP/pop) x (Energy/GDP) x (CO2/Energy)` — fuel-switching is the last term (carbon intensity of energy at fixed demand), which separates the *behavioural* mechanism from the emissions accounting identity. Price node deferred (multi-week data project, extra identification assumption).

## Step 2 — Baseline hierarchical model

Country partial-pooling on the treatment effect. **Deliberately NOT yet causal** — no country intercepts / year effects yet (added next). Purpose: learn and verify the hierarchical machinery.

$$y_i \sim \mathrm{Normal}(\alpha + \beta_{c[i]}\cdot \mathrm{has\_tax}_i,\ \sigma)$$
$$\beta_c \sim \mathrm{Normal}(\mu, \tau)$$

Priors (weakly-informative; centered at no-effect so the data, not us, moves $\mu$):
`alpha ~ Normal(0, 0.5)`, `mu ~ Normal(0, 0.5)`, `tau ~ HalfNormal(0.5)`, `sigma ~ HalfNormal(1.0)`.
Locations -> Normal; scales (SDs) -> positive-only HalfNormal.

In [1]:
import os
os.environ['PYTENSOR_FLAGS'] = 'cxx='   # PyTensor C backend broken on Windows/py3.13; nutpie compiles via numba

import pandas as pd
import numpy as np
import pymc as pm
import arviz as az

In [2]:
df = pd.read_csv('../data/cleaned/final_analysis_data.csv')

country_idx, country_labels = pd.factorize(df['country'])   # row -> int 0..162 (realizes the c[i] lookup)
n_countries = len(country_labels)
y   = df['co2_per_capita_future_trend'].values
tax = df['has_tax'].values

print(f"obs: {len(df)}  countries: {n_countries}  treated rows (has_tax): {int(tax.sum())}")

obs: 4218  countries: 163  treated rows (has_tax): 261


**Non-centered hierarchy.** Writing `beta ~ Normal(mu, tau)` directly creates *Neal's funnel* — the region the betas can occupy depends on `tau`, so the sampler stalls (`tau` got ESS=36, R-hat=1.08 in the centered version). The fix: sample a standardized `z ~ Normal(0,1)` and rebuild `beta = mu + tau*z`. Statistically identical (`mu + tau*z ~ Normal(mu, tau)`), but the geometry is decoupled so the sampler glides.

In [3]:
with pm.Model() as baseline:
    alpha = pm.Normal('alpha', mu=0, sigma=0.5)
    mu    = pm.Normal('mu',    mu=0, sigma=0.5)
    tau   = pm.HalfNormal('tau',   sigma=0.5)
    sigma = pm.HalfNormal('sigma', sigma=1.0)

    z     = pm.Normal('z', mu=0, sigma=1, shape=n_countries)   # non-centered
    beta  = pm.Deterministic('beta', mu + tau * z)             # beta_c = mu + tau*z_c ~ Normal(mu, tau)

    mu_i  = alpha + beta[country_idx] * tax
    y_obs = pm.Normal('y_obs', mu=mu_i, sigma=sigma, observed=y)

In [4]:
with baseline:
    idata = pm.sample(draws=1000, tune=1000, chains=4, target_accept=0.9,
                      random_seed=42, nuts_sampler='nutpie', progressbar=False)

NUTS[nutpie]: [alpha, mu, tau, sigma, z]


**Diagnostics first, interpretation second.** Want R-hat ≈ 1.00 (chains agree) and ESS in the hundreds+ (enough effectively-independent draws), with 0 divergences.

In [5]:
print("Convergence (top-level params):")
print(az.summary(idata, var_names=['alpha', 'mu', 'tau', 'sigma'], round_to=4).to_string())
print(f"\nDivergences: {int(idata.sample_stats['diverging'].sum())}")

post_mu = idata.posterior['mu'].values.flatten()
print(f"mu posterior mean: {post_mu.mean():.4f}   P(mu < 0): {(post_mu < 0).mean():.3f}")

Convergence (top-level params):
         mean      sd  eti89_lb  eti89_ub    ess_bulk   ess_tail   r_hat  mcse_mean  mcse_sd
alpha -0.0079  0.0054   -0.0165    0.0007   9271.2929  3084.2262  1.0001     0.0001   0.0000
mu    -0.1640  0.0305   -0.2125   -0.1166   3277.9440  2948.1450  1.0009     0.0005   0.0004
tau    0.0896  0.0387    0.0252    0.1535    693.1524   631.8018  1.0054     0.0014   0.0011
sigma  0.3482  0.0037    0.3421    0.3541  10202.2983  2956.4865  1.0012     0.0000   0.0000

Divergences: 0
mu posterior mean: -0.1640   P(mu < 0): 1.000


**Read.** All R-hat ≈ 1.00, ESS in the hundreds-to-thousands, 0 divergences — converged. `mu` ≈ −0.16 with `P(mu<0)` ≈ 1.0 echoes the Phase-1/2 ATT.

**Do NOT report this as causal yet.** The model has no country intercepts and no year effects, so `mu` absorbs cross-country level differences and global time trends, not just the policy. Next step: add country intercepts `alpha_c` (same partial-pooling trick on the baseline level) + year effects to recover a DiD-style causal estimate, then anchor priors to Phase 1/2 and splice in the mechanism layer.

## Step 3 — Make it causal: two-way fixed effects

The baseline above is **not causal**: with a single global intercept, `mu` absorbs *pre-existing, time-invariant differences between taxed and untaxed countries* (taxed countries are richer, higher-governance, already declining) plus *global time trends*. That's the "Pooled OLS without FE → omitted variable bias" trap.

The fix is the same two-way fixed effects we used in the Phase-1 DiD (`C(country) + C(year)`), now expressed as **partial-pooled random effects**:

- **country intercepts** $\alpha_c$ remove time-invariant between-country confounders → $\beta$ is identified from *within-country* variation (same country, tax-on vs tax-off).
- **year effects** $\gamma_t$ remove time-varying confounders *common to all countries* (recessions, global renewables trend) → $\beta$ is net-of-the-common-time-path.

$$y_i \sim \mathrm{Normal}(\alpha_{c[i]} + \gamma_{t[i]} + \beta_{c[i]}\cdot \mathrm{has\_tax}_i,\ \sigma)$$
$$\alpha_c \sim \mathrm{Normal}(\mu_\alpha,\ \sigma_\alpha) \qquad \gamma_t \sim \mathrm{Normal}(0,\ \sigma_\gamma) \qquad \beta_c \sim \mathrm{Normal}(\mu,\ \tau)$$

**Identification anchor.** $\alpha_c$ gets a *free* grand mean $\mu_\alpha$ (it replaces the old global `alpha`), but $\gamma_t$'s mean is *pinned at 0*. Otherwise the additive degeneracy bites: add a constant $k$ to every $\alpha_c$ and subtract it from every $\gamma_t$ and the fit is identical — the sampler drifts along that ridge. Pinning the year-effect mean nails the level into the country intercepts.

**What this does NOT buy.** Two-way FE removes those two confounder types but not time-varying confounders *specific to taxing countries* — that residual is what **parallel trends** assumes away (stress-tested in Phase 1 via event study + Rambachan–Roth). FE gives the DiD estimand *conditional on* that assumption; it doesn't make it true.

In [6]:
year_idx, year_labels = pd.factorize(df['year'])   # row -> int 0..n_years-1
n_years = len(year_labels)
print(f"years: {n_years}  ({year_labels.min()}-{year_labels.max()})")

years: 26  (1996-2021)


In [7]:
with pm.Model() as did_model:
    # country baselines (partial-pooled FE) -- free grand mean mu_a replaces the old global alpha
    mu_a    = pm.Normal('mu_a', mu=0, sigma=0.5)
    sigma_a = pm.HalfNormal('sigma_a', sigma=0.5)
    z_a     = pm.Normal('z_a', mu=0, sigma=1, shape=n_countries)
    alpha   = pm.Deterministic('alpha', mu_a + sigma_a * z_a)        # alpha_c = mu_a + sigma_a*z_a

    # year effects (partial-pooled) -- mean pinned at 0 so the level lives in alpha
    sigma_g = pm.HalfNormal('sigma_g', sigma=0.5)
    z_g     = pm.Normal('z_g', mu=0, sigma=1, shape=n_years)
    gamma   = pm.Deterministic('gamma', sigma_g * z_g)               # mean 0 by construction

    # heterogeneous tax effect (unchanged from the baseline)
    mu    = pm.Normal('mu', mu=0, sigma=0.5)
    tau   = pm.HalfNormal('tau', sigma=0.5)
    z     = pm.Normal('z', mu=0, sigma=1, shape=n_countries)
    beta  = pm.Deterministic('beta', mu + tau * z)

    sigma = pm.HalfNormal('sigma', sigma=1.0)

    mu_i  = alpha[country_idx] + gamma[year_idx] + beta[country_idx] * tax
    y_obs = pm.Normal('y_obs', mu=mu_i, sigma=sigma, observed=y)

In [8]:
with did_model:
    idata_did = pm.sample(draws=1000, tune=1000, chains=4, target_accept=0.9,
                          random_seed=42, nuts_sampler='nutpie', progressbar=False)

NUTS[nutpie]: [mu_a, sigma_a, z_a, sigma_g, z_g, mu, tau, z, sigma]


In [9]:
print("Convergence (top-level params):")
print(az.summary(idata_did, var_names=['mu', 'tau', 'mu_a', 'sigma_a', 'sigma_g', 'sigma'],
                 round_to=4).to_string())
print(f"\nDivergences: {int(idata_did.sample_stats['diverging'].sum())}")

post_mu = idata_did.posterior['mu'].values.flatten()
print(f"mu posterior mean: {post_mu.mean():.4f}   P(mu < 0): {(post_mu < 0).mean():.3f}")

Convergence (top-level params):
           mean      sd  eti89_lb  eti89_ub   ess_bulk   ess_tail   r_hat  mcse_mean  mcse_sd
mu      -0.1214  0.0349   -0.1762   -0.0669  1877.6440  2354.8240  1.0013     0.0008   0.0006
tau      0.0864  0.0442    0.0173    0.1582   526.3971   611.6380  1.0066     0.0018   0.0014
mu_a    -0.0096  0.0152   -0.0336    0.0146   628.6630   974.9786  1.0037     0.0006   0.0004
sigma_a  0.1119  0.0083    0.0990    0.1258  1284.1217  1533.2119  1.0005     0.0002   0.0002
sigma_g  0.0551  0.0103    0.0405    0.0731  1049.5173  1550.9827  1.0085     0.0003   0.0003
sigma    0.3265  0.0036    0.3207    0.3324  5976.4660  2934.5931  1.0016     0.0000   0.0000

Divergences: 0
mu posterior mean: -0.1214   P(mu < 0): 0.999


**Read.** Converged (R-hat ≈ 1.00, ESS in the hundreds-to-thousands, 0 divergences). `mu` moved **−0.164 → −0.121**: the ~0.04 shift toward zero is the confounding the FE absorbed (`sigma_a` ≈ 0.11 — countries differ in baseline; `sigma_g` ≈ 0.055 — modest common year shifts), and `sigma` dropped because those structured effects explain real variance. What's left, **−0.121**, is the within-country, net-of-common-time DiD estimand.

**Validation:** −0.121 lands on the Phase-1/2 ATT (≈ −0.13). Frequentist two-way-FE DiD and this Bayesian partial-pooled model — different machines, same identification, same number.

**Caveats.** (1) `has_tax` only; Phase 2 found **ETS carries the effect** — next step adds `has_ets` as a second treatment, after which `mu` reads as "tax holding ETS fixed." (2) `mu` is the population mean of the heterogeneous `beta_c` (`tau` ≈ 0.086 spread); that spread is what will drive country-specific recommendations later.

## Step 4 — Second treatment: de-confound tax from ETS

`has_tax` and `has_ets` **co-occur** — most EU countries run both (EU ETS since 2005 + national carbon taxes). A tax-only model leaves ETS out, and because ETS also lowers emissions *and* is correlated with tax adoption, the tax coefficient **absorbs ETS's effect** (omitted-treatment bias). So Step 3's `mu = -0.121` is partly ETS leaking in.

This is the Phase-2 de-conflation of `post_carbon_tax` → clean `has_tax`/`has_ets`, now reproduced *inside* the Bayesian model by giving ETS its own coefficient:

$$y_i \sim \mathrm{Normal}\big(\alpha_{c[i]} + \gamma_{t[i]} + \beta_{c[i]}\cdot \mathrm{has\_tax}_i + \mu_{ets}\cdot \mathrm{has\_ets}_i,\ \sigma\big)$$

`mu_ets` is a **single pooled** coefficient for this increment (deferred: promote to heterogeneous `beta^{ets}_c`, and add the tax×ETS interaction — Phase 2's "both" cell). One new idea at a time: confirm the de-confounding echo first.

In [10]:
ets = df['has_ets'].values
print(f"treated rows -- has_tax: {int(tax.sum())}  has_ets: {int(ets.sum())}")

treated rows -- has_tax: 261  has_ets: 469


In [11]:
with pm.Model() as did_ets:
    # country baselines (partial-pooled FE) -- free grand mean
    mu_a    = pm.Normal('mu_a', mu=0, sigma=0.5)
    sigma_a = pm.HalfNormal('sigma_a', sigma=0.5)
    z_a     = pm.Normal('z_a', mu=0, sigma=1, shape=n_countries)
    alpha   = pm.Deterministic('alpha', mu_a + sigma_a * z_a)

    # year effects (partial-pooled) -- mean pinned at 0
    sigma_g = pm.HalfNormal('sigma_g', sigma=0.5)
    z_g     = pm.Normal('z_g', mu=0, sigma=1, shape=n_years)
    gamma   = pm.Deterministic('gamma', sigma_g * z_g)

    # heterogeneous tax effect
    mu    = pm.Normal('mu', mu=0, sigma=0.5)
    tau   = pm.HalfNormal('tau', sigma=0.5)
    z     = pm.Normal('z', mu=0, sigma=1, shape=n_countries)
    beta  = pm.Deterministic('beta', mu + tau * z)

    # pooled ETS effect (single coefficient -- promote to heterogeneous later)
    mu_ets = pm.Normal('mu_ets', mu=0, sigma=0.5)

    sigma = pm.HalfNormal('sigma', sigma=1.0)

    mu_i  = alpha[country_idx] + gamma[year_idx] + beta[country_idx] * tax + mu_ets * ets
    y_obs = pm.Normal('y_obs', mu=mu_i, sigma=sigma, observed=y)

In [12]:
with did_ets:
    idata_ets = pm.sample(draws=1000, tune=1000, chains=4, target_accept=0.9,
                          random_seed=42, nuts_sampler='nutpie', progressbar=False)

NUTS[nutpie]: [mu_a, sigma_a, z_a, sigma_g, z_g, mu, tau, z, mu_ets, sigma]


In [13]:
print("Convergence (top-level params):")
print(az.summary(idata_ets, var_names=['mu', 'mu_ets', 'tau', 'mu_a', 'sigma_a', 'sigma_g', 'sigma'],
                 round_to=4).to_string())
print(f"\nDivergences: {int(idata_ets.sample_stats['diverging'].sum())}")

post_mu  = idata_ets.posterior['mu'].values.flatten()
post_ets = idata_ets.posterior['mu_ets'].values.flatten()
print(f"mu (tax)  mean: {post_mu.mean():.4f}   P(<0): {(post_mu  < 0).mean():.3f}")
print(f"mu_ets    mean: {post_ets.mean():.4f}   P(<0): {(post_ets < 0).mean():.3f}")

Convergence (top-level params):
           mean      sd  eti89_lb  eti89_ub   ess_bulk   ess_tail   r_hat  mcse_mean  mcse_sd
mu      -0.0677  0.0357   -0.1230   -0.0102  1762.4679  2212.6627  1.0017     0.0009   0.0006
mu_ets  -0.1649  0.0236   -0.2030   -0.1272  1879.6809  2446.6048  1.0004     0.0005   0.0004
tau      0.0866  0.0432    0.0156    0.1557   445.6315   402.8621  1.0106     0.0019   0.0012
mu_a     0.0054  0.0139   -0.0166    0.0270   736.9452  1073.2280  1.0054     0.0005   0.0004
sigma_a  0.1062  0.0081    0.0938    0.1195  1547.1409  2112.6310  1.0012     0.0002   0.0001
sigma_g  0.0472  0.0092    0.0342    0.0628  1067.9706  1815.2944  1.0042     0.0003   0.0003
sigma    0.3254  0.0036    0.3196    0.3312  4929.2327  2885.5088  1.0012     0.0001   0.0000

Divergences: 0
mu (tax)  mean: -0.0677   P(<0): 0.971
mu_ets    mean: -0.1649   P(<0): 1.000


**Read.** Converged (R-hat ≈ 1.00, 0 divergences). Giving ETS its own coefficient nearly **halves the tax effect** and reproduces the Phase-2 verdict:

| | Step 3 (tax only) | Step 4 (tax + ETS) | Phase 2 (frequentist) |
|---|---|---|---|
| mu_tax | −0.121 | **−0.068** (P<0 = 0.97) | −0.056 (n.s.) |
| mu_ets | — | **−0.165** (P<0 = 1.00) | −0.13 (sig) |

The ~0.05 the tax coefficient shed is the ETS effect that had been leaking into it (omitted-treatment bias). `mu_ets` is strong and precise (more ETS rows, and pooled). **"Carbon pricing works; it's the ETS not the tax"** — the Phase-2 headline now falls out of the hierarchical model too. `mu_tax` reads honestly as the tax effect *holding ETS and the fixed effects fixed*: weak.

**Implication for the engine:** lean on the ETS lever for predicted reductions; treat a standalone carbon tax as a marginal mover.

**Deferred (backlog):** heterogeneous `beta^{ets}_c`; tax×ETS interaction (Phase 2's "both" cell, which was sub-additive).

## Step 5 — Anchor the priors (honestly)

So far every prior is weakly-informative at zero (`Normal(0, 0.5)`, `HalfNormal(0.5)`) — the model rediscovers everything from scratch. We *know* more than that (Phase 1/2: effects are modest and negative, ETS carries it). The prior is where that knowledge belongs.

**The trap.** Phase 1/2 estimated ATT ≈ −0.13 *from this same dataset*. Centering a prior on −0.13 and re-fitting on the same rows uses the data **twice** — once to build the prior, once in the likelihood — producing a posterior that is **artificially overconfident** (falsely narrow CIs). A prior must be *independent* of the likelihood data; our own same-data estimate is not.

**The honest fix.** Anchor to information that *isn't this dataset*:
- **Center stays at 0; tighten the scale.** The robust external knowledge is that effects are *modest* (an effect of 0.5 in these units is ~1.4σ — absurd). `Normal(0, 0.2)` encodes "small effects," which the whole carbon-pricing literature supports independently of our data. Center at 0 → the **sign is earned from data, not assumed**.
- **Regularize the hierarchy scales** (`τ`, `σ_α`, `σ_γ`) — tighter `τ` ⇒ stronger pooling ⇒ noisy per-country `β_c` shrink toward the mean. This is the regularization the recommendation engine needs.
- **Phase 1/2 becomes the *validation*, not the input** — if the posterior still lands near −0.13/−0.16 under an honest prior, that's a genuine consistency check.

We fit **loose vs anchored** side by side to see two things: (1) do the top-level estimates move? (they shouldn't — n=4,218 means data dominates the prior); (2) does the country-level `β_c` spread shrink? (it should — that's the regularization).

In [14]:
# countries ever taxed -- the only ones whose beta_c is informed by data
treated_countries = np.unique(country_idx[tax == 1])

def make_model(hp):
    with pm.Model() as m:
        mu_a    = pm.Normal('mu_a', mu=0, sigma=0.5)
        sigma_a = pm.HalfNormal('sigma_a', sigma=hp['s_a'])
        z_a     = pm.Normal('z_a', mu=0, sigma=1, shape=n_countries)
        alpha   = pm.Deterministic('alpha', mu_a + sigma_a * z_a)

        sigma_g = pm.HalfNormal('sigma_g', sigma=hp['s_g'])
        z_g     = pm.Normal('z_g', mu=0, sigma=1, shape=n_years)
        gamma   = pm.Deterministic('gamma', sigma_g * z_g)

        mu    = pm.Normal('mu', mu=0, sigma=hp['eff'])
        tau   = pm.HalfNormal('tau', sigma=hp['tau'])
        z     = pm.Normal('z', mu=0, sigma=1, shape=n_countries)
        beta  = pm.Deterministic('beta', mu + tau * z)

        mu_ets = pm.Normal('mu_ets', mu=0, sigma=hp['eff'])
        sigma  = pm.HalfNormal('sigma', sigma=hp['resid'])

        mu_i = alpha[country_idx] + gamma[year_idx] + beta[country_idx] * tax + mu_ets * ets
        pm.Normal('y_obs', mu=mu_i, sigma=sigma, observed=y)
    return m

def fit(m):
    with m:
        return pm.sample(draws=1000, tune=1000, chains=4, target_accept=0.9,
                         random_seed=42, nuts_sampler='nutpie', progressbar=False)

In [15]:
loose    = {'eff': 0.5, 'tau': 0.5, 's_a': 0.5, 's_g': 0.5, 'resid': 1.0}
anchored = {'eff': 0.2, 'tau': 0.2, 's_a': 0.2, 's_g': 0.1, 'resid': 0.5}

idata_loose    = fit(make_model(loose))
idata_anchored = fit(make_model(anchored))

NUTS[nutpie]: [mu_a, sigma_a, z_a, sigma_g, z_g, mu, tau, z, mu_ets, sigma]


NUTS[nutpie]: [mu_a, sigma_a, z_a, sigma_g, z_g, mu, tau, z, mu_ets, sigma]


In [16]:
def report(name, idata):
    s = az.summary(idata, var_names=['mu', 'mu_ets', 'tau'], round_to=4)
    beta_means = idata.posterior['beta'].mean(('chain', 'draw')).values[treated_countries]
    div = int(idata.sample_stats['diverging'].sum())
    print(f"[{name}]  divergences={div}")
    print(s[['mean', 'sd', 'ess_bulk', 'r_hat']].to_string())
    print(f"  treated-country beta_c spread: sd={beta_means.std():.4f}  "
          f"range=[{beta_means.min():.3f}, {beta_means.max():.3f}]\n")

report('loose (step 4)', idata_loose)
report('anchored',       idata_anchored)

[loose (step 4)]  divergences=0
          mean      sd   ess_bulk   r_hat
mu     -0.0677  0.0357  1762.4679  1.0017
mu_ets -0.1649  0.0236  1879.6809  1.0004
tau     0.0866  0.0432   445.6315  1.0106
  treated-country beta_c spread: sd=0.0441  range=[-0.232, -0.014]

[anchored]  divergences=0
          mean      sd   ess_bulk   r_hat
mu     -0.0653  0.0339  1657.7089  1.0010
mu_ets -0.1632  0.0234  2180.5011  1.0007
tau     0.0822  0.0431   499.4770  1.0053
  treated-country beta_c spread: sd=0.0413  range=[-0.221, -0.018]



**Read.** Both converged, 0 divergences.

| | loose (step 4) | anchored | moved? |
|---|---|---|---|
| mu_tax | −0.068 (sd 0.036) | −0.065 (sd 0.034) | third decimal |
| mu_ets | −0.165 (sd 0.024) | −0.163 (sd 0.023) | third decimal |
| tau | 0.087 | 0.082 | barely |
| beta_c spread (sd / range) | 0.044 / [−0.232, −0.014] | 0.041 / [−0.221, −0.018] | **shrank** |

1. **Top-level estimates barely move** — tightening every effect prior 2.5× shifts `mu`/`mu_ets` in the third decimal. With n=4,218 the likelihood dominates the prior, so the headline (ETS −0.16, tax weak) is a property of the *data*, not the prior. Robustness check passed.
2. **Country-level spread shrinks** — `beta_c` tightened (sd 0.044 → 0.041, extremes pulled in). The tighter `tau` prior pools noisy per-country estimates harder. Modest here (the loose prior wasn't crazy; most treated countries have enough rows), but it's the regularization the engine needs and matters more for thin-data countries.
3. **No circularity, still validated** — we never fed −0.13 into the prior, yet `mu_ets` lands at −0.163, on the Phase-1/2 ATT. Agreement under an honest prior is a real consistency check, not a self-fulfilling one.

**The anchored prior set is our standing model going forward** (`idata_anchored`): same answer, honestly regularized, no double-counting.

## Step 6 — Structural mechanism layer (Kaya decomposition)

Everything so far is **reduced-form**: `policy → emissions`, one arrow. `mu_ets` tells us ETS lowers emissions but not *how*. The recommendation engine needs a **mechanism breakdown**, so we open the box: `policy → mechanism → emissions`.

**Scaffolding — the Kaya identity** decomposes emissions/capita into three multiplicative channels:

$$\frac{\text{CO}_2}{\text{pop}} = \underbrace{\frac{\text{GDP}}{\text{pop}}}_{\text{affluence (activity)}} \times \underbrace{\frac{\text{Energy}}{\text{GDP}}}_{\text{energy intensity (efficiency)}} \times \underbrace{\frac{\text{CO}_2}{\text{Energy}}}_{\text{carbon intensity (fuel switch)}}$$

The **fuel-switching** behavioural story lives entirely in **carbon intensity** — the channel a well-functioning carbon price *should* move (same energy, cleaner source = the "good reason" emissions fall). **Affluence** is the channel to *worry* about: if emissions fell because activity shrank or moved offshore (**leakage**), that's the "bad reason." Efficiency is a legitimate middle.

This is **Phase 1's confounder/mediator rule run in reverse**: the energy/GDP variables we *refused* to control for (because they're mediators that would block the mechanism) are exactly the pathways we now explicitly model.

**Why logs.** Kaya is multiplicative → additive in logs, so the 3-yr forward *log* change decomposes exactly: `dlog(CO2/pop) = dlog(A) + dlog(I) + dlog(K)`. (Our earlier outcome `co2_per_capita_future_trend` is a *level* trend, so it can't decompose — we switch to a log-change outcome here. Magnitudes differ from earlier steps; these are annualized log-points, ~percent per year.)

In [17]:
dfk = pd.read_csv('../data/cleaned/final_analysis_data.csv').sort_values(['country', 'year']).copy()

# Kaya level channels:  CO2/pop = A * I * K
dfk['kaya_A'] = dfk['gdp'] / dfk['population']                    # affluence  (GDP/pop)
dfk['kaya_I'] = dfk['energy_per_gdp']                            # energy intensity (Energy/GDP)
dfk['kaya_K'] = dfk['co2_per_capita'] / dfk['energy_per_capita']  # carbon intensity (CO2/Energy)
dfk['kaya_T'] = dfk['co2_per_capita']                           # total
for c in ['A', 'I', 'K', 'T']:
    dfk[f'log_{c}'] = np.log(dfk[f'kaya_{c}'])

# identity check (level): does A*I*K reproduce CO2/pop?
prod = dfk['kaya_A'] * dfk['kaya_I'] * dfk['kaya_K']
ok = dfk[['kaya_A', 'kaya_I', 'kaya_K', 'kaya_T']].notna().all(axis=1) & (dfk['kaya_T'] > 0)
rel = ((prod[ok] - dfk['kaya_T'][ok]) / dfk['kaya_T'][ok]).abs()
print(f"level identity A*I*K vs CO2/pop: max rel err={rel.max():.2e}  median={rel.median():.2e}")

level identity A*I*K vs CO2/pop: max rel err=5.92e-03  median=1.93e-04


In [18]:
# forward 3-yr annualized log change per channel (self-merge on country, year+3)
fwd = dfk[['country', 'year', 'log_A', 'log_I', 'log_K', 'log_T']].copy()
fwd['year'] = fwd['year'] - 3
fwd = fwd.rename(columns={f'log_{c}': f'log_{c}_p3' for c in ['A', 'I', 'K', 'T']})
dfk = dfk.merge(fwd, on=['country', 'year'], how='left')
for c in ['A', 'I', 'K', 'T']:
    dfk[f'd_{c}'] = (dfk[f'log_{c}_p3'] - dfk[f'log_{c}']) / 3.0

# decomposition closes? d_T == d_A + d_I + d_K
have = dfk[['d_A', 'd_I', 'd_K', 'd_T']].notna().all(axis=1)
resid = (dfk['d_T'][have] - (dfk['d_A'][have] + dfk['d_I'][have] + dfk['d_K'][have])).abs()
print(f"log decomposition d_T vs d_A+d_I+d_K: max resid={resid.max():.2e}  median={resid.median():.2e}")

log decomposition d_T vs d_A+d_I+d_K: max resid=3.53e-03  median=8.10e-05


In [19]:
# common mechanism sample: all channels + treatments present
mk = dfk[dfk[['d_A', 'd_I', 'd_K', 'd_T', 'has_tax', 'has_ets']].notna().all(axis=1)].copy()
c_idx, c_lab = pd.factorize(mk['country'])
t_idx, t_lab = pd.factorize(mk['year'])
nc, ny = len(c_lab), len(t_lab)
tax_k, ets_k = mk['has_tax'].values, mk['has_ets'].values
print(f"mechanism sample: {len(mk)} rows, {nc} countries")

def fit_channel(yvals):
    # anchored priors; country+year FE; BOTH treatments pooled for a clean decomposition
    with pm.Model():
        mu_a    = pm.Normal('mu_a', 0, 0.5)
        sigma_a = pm.HalfNormal('sigma_a', 0.2)
        alpha   = mu_a + sigma_a * pm.Normal('z_a', 0, 1, shape=nc)
        sigma_g = pm.HalfNormal('sigma_g', 0.1)
        gamma   = sigma_g * pm.Normal('z_g', 0, 1, shape=ny)
        mu_tax  = pm.Normal('mu_tax', 0, 0.2)
        mu_ets  = pm.Normal('mu_ets', 0, 0.2)
        sigma   = pm.HalfNormal('sigma', 0.2)
        mu_i = alpha[c_idx] + gamma[t_idx] + mu_tax * tax_k + mu_ets * ets_k
        pm.Normal('y_obs', mu=mu_i, sigma=sigma, observed=yvals)
        idata = pm.sample(draws=1000, tune=1000, chains=4, target_accept=0.9,
                          random_seed=42, nuts_sampler='nutpie', progressbar=False)
    return (idata.posterior['mu_tax'].values.flatten(),
            idata.posterior['mu_ets'].values.flatten(),
            int(idata.sample_stats['diverging'].sum()))

mechanism sample: 3728 rows, 163 countries


In [20]:
labels = {'T': 'TOTAL', 'K': 'carbon intensity (fuel switch)',
          'I': 'energy intensity (efficiency)', 'A': 'affluence (activity)'}
res = {}
for c in ['T', 'K', 'I', 'A']:
    t, e, div = fit_channel(mk[f'd_{c}'].values)
    res[c] = (t, e)
    print(f"[{labels[c]:32s}] div={div}  "
          f"mu_tax={t.mean():+.4f} (P<0={(t<0).mean():.2f})  "
          f"mu_ets={e.mean():+.4f} (P<0={(e<0).mean():.2f})")

st = sum(res[c][0].mean() for c in ['K', 'I', 'A'])
se = sum(res[c][1].mean() for c in ['K', 'I', 'A'])
print(f"\nclosure tax: total={res['T'][0].mean():+.4f}  K+I+A={st:+.4f}")
print(f"closure ets: total={res['T'][1].mean():+.4f}  K+I+A={se:+.4f}")

NUTS[nutpie]: [mu_a, sigma_a, z_a, sigma_g, z_g, mu_tax, mu_ets, sigma]


[TOTAL                           ] div=0  mu_tax=-0.0133 (P<0=0.99)  mu_ets=-0.0286 (P<0=1.00)


NUTS[nutpie]: [mu_a, sigma_a, z_a, sigma_g, z_g, mu_tax, mu_ets, sigma]


[carbon intensity (fuel switch)  ] div=0  mu_tax=-0.0062 (P<0=0.86)  mu_ets=-0.0143 (P<0=1.00)


NUTS[nutpie]: [mu_a, sigma_a, z_a, sigma_g, z_g, mu_tax, mu_ets, sigma]


[energy intensity (efficiency)   ] div=0  mu_tax=-0.0121 (P<0=0.97)  mu_ets=-0.0020 (P<0=0.67)


NUTS[nutpie]: [mu_a, sigma_a, z_a, sigma_g, z_g, mu_tax, mu_ets, sigma]


[affluence (activity)            ] div=0  mu_tax=+0.0094 (P<0=0.00)  mu_ets=-0.0093 (P<0=1.00)

closure tax: total=-0.0133  K+I+A=-0.0090
closure ets: total=-0.0286  K+I+A=-0.0256


**Read (annualized log-points; ×3 ≈ window %, log-point ≈ percent).**

| channel | mu_ets/yr | mu_tax/yr |
|---|---|---|
| TOTAL | −0.0286 (P<0=1.00) | −0.0133 (P<0=0.99) |
| ✅ carbon intensity (fuel switch) | **−0.0143** (P<0=1.00) | −0.0062 (P<0=0.86) |
| 🟡 energy intensity (efficiency) | −0.0020 (P<0=0.67) | −0.0121 (P<0=0.97) |
| ⚠️ affluence (activity) | −0.0093 (P<0=1.00) | +0.0094 (P<0=0.00) |

- **ETS:** ~half its effect is real **fuel-switching** (carbon intensity, strong) — the good channel — but ~a third rides on **lower activity** (affluence) — the worry channel (leakage / suppressed activity). Efficiency ≈ 0. Over a 3-yr window: ≈ −8% total ≈ −4% decarbonization + −3% activity. The engine should flag the activity share.
- **Carbon tax:** weak total, and what it does runs through **efficiency**, not fuel-switching (carbon intensity uncertain, P<0=0.86); its affluence channel is *positive* (taxing countries grew, masking the effect). Mechanistic echo of Phase 2's null outcome-sensitivity.

**Caveats.** (1) Closure ~85–90%, not exact (OLS would close to machine precision; priors + partial-pooled FE shrink each channel independently). Trust the *shape*, not the last decimal. (2) The affluence channel is an *association* within the DiD, not a proven causal mechanism — mediation assumes no mediator–outcome confounding (strong). (3) Pooled treatments (heterogeneity set aside for a clean split).